In [43]:
import pandas as pd

df = pd.read_csv(
    'Data/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv',
    low_memory=False
)

df.head()

,DESYNPUF_ID,PDE_ID,SRVC_DT,PROD_SRVC_ID,QTY_DSPNSD_NUM,DAYS_SUPLY_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT
0,00013D2EFD8E45D1,233664490397622,20080103,00247037252,30.0,20,10.0,120.0
1,00013D2EFD8E45D1,233644490171972,20080105,00223039502,10.0,10,0.0,0.0
2,00013D2EFD8E45D1,233974489116848,20080109,00364724812,120.0,30,10.0,110.0
3,00013D2EFD8E45D1,233574491083209,20080123,00179180672,30.0,30,0.0,240.0
4,00013D2EFD8E45D1,233024491180571,20080124,58016005300,30.0,30,70.0,70.0


In [44]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(5552421, 8)
DESYNPUF_ID        object
PDE_ID              int64
SRVC_DT             int64
PROD_SRVC_ID       object
QTY_DSPNSD_NUM    float64
DAYS_SUPLY_NUM      int64
PTNT_PAY_AMT      float64
TOT_RX_CST_AMT    float64
dtype: object
DESYNPUF_ID       0
PDE_ID            0
SRVC_DT           0
PROD_SRVC_ID      0
QTY_DSPNSD_NUM    0
DAYS_SUPLY_NUM    0
PTNT_PAY_AMT      0
TOT_RX_CST_AMT    0
dtype: int64


In [45]:
df['SRVC_DT'] = pd.to_datetime(df['SRVC_DT'], format='%Y%m%d')

# Verify it worked
print(df['SRVC_DT'].dtype)
print(df['SRVC_DT'].min(), "→", df['SRVC_DT'].max())

df.head()

datetime64[ns]
2008-01-01 00:00:00 → 2010-12-31 00:00:00


,DESYNPUF_ID,PDE_ID,SRVC_DT,PROD_SRVC_ID,QTY_DSPNSD_NUM,DAYS_SUPLY_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT
0,00013D2EFD8E45D1,233664490397622,2008-01-03,00247037252,30.0,20,10.0,120.0
1,00013D2EFD8E45D1,233644490171972,2008-01-05,00223039502,10.0,10,0.0,0.0
2,00013D2EFD8E45D1,233974489116848,2008-01-09,00364724812,120.0,30,10.0,110.0
3,00013D2EFD8E45D1,233574491083209,2008-01-23,00179180672,30.0,30,0.0,240.0
4,00013D2EFD8E45D1,233024491180571,2008-01-24,58016005300,30.0,30,70.0,70.0


In [46]:
# Count how many fills each patient-drug pair has
pair_counts = df.groupby(['DESYNPUF_ID', 'PROD_SRVC_ID']).size()

# How many pairs have more than 1 fill (i.e., repeated purchases)?
repeated = pair_counts[pair_counts > 1]
not_repeated = pair_counts[pair_counts == 1]

print(f"Total unique patient-drug pairs:       {len(pair_counts):,}")
print(f"Pairs with ONLY 1 fill (no repeat):    {len(not_repeated):,}")
print(f"Pairs with 2+ fills (repeated):        {len(repeated):,}")
print(f"Percentage with repeated purchases:    {len(repeated)/len(pair_counts)*100:.1f}%")

# Distribution of how many fills per pair
print("\nFill count distribution:")
print(pair_counts.value_counts().sort_index().head(10))

Total unique patient-drug pairs:       5,549,151
Pairs with ONLY 1 fill (no repeat):    5,545,888
Pairs with 2+ fills (repeated):        3,263
Percentage with repeated purchases:    0.1%

Fill count distribution:
1    5545888
2       3256
3          7
Name: count, dtype: int64


In [47]:
# Extract the 4 middle digits = ingredient, strength, dosage form & route
df['DRUG_MID_3'] = df['PROD_SRVC_ID'].astype(str).str[5:8]

pair_counts_broad = df.groupby(['DESYNPUF_ID', 'DRUG_MID_3']).size()
repeated_broad = pair_counts_broad[pair_counts_broad > 1]
print(f"Pairs with 2+ fills: {len(repeated_broad):,}")

Pairs with 2+ fills: 826,615


In [48]:
# Cell left for testing different drug code groupings, e.g., by the last 3 digits instead of the middle 4
# Get the last 3 digits of the drug code to see if certain drug classes have more repeats
#df['DRUG_3'] = df['PROD_SRVC_ID'].astype(str).str[-3:]
# Groups all rows by patient + the last 3 digits of the drug code, then counts how many fills each group has
#pair_counts_broad = df.groupby(['DESYNPUF_ID', 'DRUG_3']).size()
# filter to only the pairs that have more than 1 fill
#repeated_broad = pair_counts_broad[pair_counts_broad > 1]
# print the total count of patient-drug pairs that have repeated fills
#print(f"Pairs with 2+ fills: {len(repeated_broad):,}")

In [49]:
# Sort the DataFrame by DESYNPUF_ID, PROD_SRVC_ID last 3 digits (DRUG_3), and SRVC_DT to ensure the runout date is calculated correctly 
df = df.sort_values(['DESYNPUF_ID', 'DRUG_MID_3', 'SRVC_DT']).reset_index(drop=True)

# Calculate the expected runout date by summing up the service date and the days of suply
df['runout_date'] = df['SRVC_DT'] + pd.to_timedelta(df['DAYS_SUPLY_NUM'], unit='D')

# Get the next fill date for the same patient + drug group
df['next_fill_date'] = df.groupby(['DESYNPUF_ID', 'DRUG_MID_3'])['SRVC_DT'].shift(-1)

df.head(30)

,DESYNPUF_ID,PDE_ID,SRVC_DT,PROD_SRVC_ID,QTY_DSPNSD_NUM,DAYS_SUPLY_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT,DRUG_MID_3,runout_date,next_fill_date
0,00013D2EFD8E45D1,233784494196079,2008-02-16,50070000265,360.0,30,0.0,10.0,000,2008-03-17,2008-11-27
1,00013D2EFD8E45D1,233924493383615,2008-11-27,00799000105,10.0,30,0.0,160.0,000,2008-12-27,2010-02-24
2,00013D2EFD8E45D1,233124489977780,2010-02-24,49575000101,30.0,30,10.0,20.0,000,2010-03-26,NaT
3,00013D2EFD8E45D1,233884492803278,2008-01-24,53650001801,30.0,30,10.0,40.0,001,2008-02-23,NaT
4,00013D2EFD8E45D1,233264492055411,2008-11-14,37000003201,30.0,10,0.0,10.0,003,2008-11-24,2009-06-26
5,00013D2EFD8E45D1,233814493968069,2009-06-26,54441003225,20.0,30,0.0,0.0,003,2009-07-26,NaT
6,00013D2EFD8E45D1,233644488961516,2008-12-03,61916004301,90.0,30,0.0,10.0,004,2009-01-02,2008-12-28
7,00013D2EFD8E45D1,233344494226844,2008-12-28,65427004630,30.0,30,20.0,280.0,004,2009-01-27,2010-08-28
8,00013D2EFD8E45D1,233824490977673,2010-08-28,61392004154,180.0,30,0.0,10.0,004,2010-09-27,2010-08-31
9,00013D2EFD8E45D1,233964490644916,2010-08-31,16590004660,90.0,30,0.0,10.0,004,2010-09-30,2010-09-01


In [50]:
# Define the columns we care about first
priority_cols = ['DESYNPUF_ID', 'DRUG_MID_3', 'SRVC_DT', 'DAYS_SUPLY_NUM', 
                 'runout_date', 'next_fill_date']

# Everything else goes to the right
other_cols = [col for col in df.columns if col not in priority_cols]

# Reorder
df = df[priority_cols + other_cols]

df.head(30)

,DESYNPUF_ID,DRUG_MID_3,SRVC_DT,DAYS_SUPLY_NUM,runout_date,next_fill_date,PDE_ID,PROD_SRVC_ID,QTY_DSPNSD_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT
0,00013D2EFD8E45D1,000,2008-02-16,30,2008-03-17,2008-11-27,233784494196079,50070000265,360.0,0.0,10.0
1,00013D2EFD8E45D1,000,2008-11-27,30,2008-12-27,2010-02-24,233924493383615,00799000105,10.0,0.0,160.0
2,00013D2EFD8E45D1,000,2010-02-24,30,2010-03-26,NaT,233124489977780,49575000101,30.0,10.0,20.0
3,00013D2EFD8E45D1,001,2008-01-24,30,2008-02-23,NaT,233884492803278,53650001801,30.0,10.0,40.0
4,00013D2EFD8E45D1,003,2008-11-14,10,2008-11-24,2009-06-26,233264492055411,37000003201,30.0,0.0,10.0
5,00013D2EFD8E45D1,003,2009-06-26,30,2009-07-26,NaT,233814493968069,54441003225,20.0,0.0,0.0
6,00013D2EFD8E45D1,004,2008-12-03,30,2009-01-02,2008-12-28,233644488961516,61916004301,90.0,0.0,10.0
7,00013D2EFD8E45D1,004,2008-12-28,30,2009-01-27,2010-08-28,233344494226844,65427004630,30.0,20.0,280.0
8,00013D2EFD8E45D1,004,2010-08-28,30,2010-09-27,2010-08-31,233824490977673,61392004154,180.0,0.0,10.0
9,00013D2EFD8E45D1,004,2010-08-31,30,2010-09-30,2010-09-01,233964490644916,16590004660,90.0,0.0,10.0


In [51]:
# Calculate how many days late the next fill was (positive = late, negative = early)
df['refill_gap_days'] = (df['next_fill_date'] - df['runout_date']).dt.days

# Label: late if next fill is more than 7 days after run-out
GRACE_WINDOW = 7
df['is_late'] = (df['refill_gap_days'] > GRACE_WINDOW).astype(int)

# Drop rows where there is no next fill (last fill per patient-drug — can't be labelled)
df_labelled = df.dropna(subset=['next_fill_date']).copy()

print(f"Total labelled rows: {len(df_labelled):,}")
print(f"\nClass balance:")
print(df_labelled['is_late'].value_counts())
print(df_labelled['is_late'].value_counts(normalize=True).round(3))

Total labelled rows: 1,142,880

Class balance:
is_late
1    1008425
0     134455
Name: count, dtype: int64
is_late
1    0.882
0    0.118
Name: proportion, dtype: float64


In [56]:
# Sort df_labelled to ensure correct ordering for feature calculations
df_labelled = df_labelled.sort_values(['DESYNPUF_ID', 'DRUG_MID_3', 'SRVC_DT']).reset_index(drop=True)

# Feature 1: refill gap from previous fill (how late/early was the LAST refill?)
df_labelled['prev_gap_days'] = df_labelled.groupby(
    ['DESYNPUF_ID', 'DRUG_MID_3'])['refill_gap_days'].shift(1)

# Feature 2: how many fills has this patient had for this drug so far?
df_labelled['fill_count'] = df_labelled.groupby(
    ['DESYNPUF_ID', 'DRUG_MID_3']).cumcount() + 1

# Feature 3: days of supply (already a column, useful as a feature)
# Feature 4: pa tient pay amount and total cost (already columns)

print(df_labelled[['DESYNPUF_ID', 'DRUG_MID_3', 'refill_gap_days', 
                    'prev_gap_days', 'fill_count', 'is_late']].head(50))

         DESYNPUF_ID DRUG_MID_3  refill_gap_days  prev_gap_days  fill_count  \
0   00013D2EFD8E45D1        000            255.0            NaN           1   
1   00013D2EFD8E45D1        000            424.0          255.0           2   
2   00013D2EFD8E45D1        003            214.0            NaN           1   
3   00013D2EFD8E45D1        004             -5.0            NaN           1   
4   00013D2EFD8E45D1        004            578.0           -5.0           2   
5   00013D2EFD8E45D1        004            -27.0          578.0           3   
6   00013D2EFD8E45D1        004            -29.0          -27.0           4   
7   00013D2EFD8E45D1        004            -22.0          -29.0           5   
8   00013D2EFD8E45D1        005            229.0            NaN           1   
9   00013D2EFD8E45D1        005            156.0          229.0           2   
10  00013D2EFD8E45D1        005            154.0          156.0           3   
11  00013D2EFD8E45D1        006            790.0    

In [53]:
# Count NaN in prev_gap_days
total_rows = len(df_labelled)
nan_count = df_labelled['prev_gap_days'].isna().sum()
pct = nan_count / total_rows * 100

print(f"Total rows:        {total_rows:,}")
print(f"NaN in prev_gap:   {nan_count:,}  ({pct:.1f}%)")
print(f"Rows kept:         {total_rows - nan_count:,}  ({100 - pct:.1f}%)")

Total rows:        1,142,880
NaN in prev_gap:   826,615  (72.3%)
Rows kept:         316,265  (27.7%)


In [62]:
# Recalculate prev_gap_days cleanly
df_labelled['prev_gap_days'] = df_labelled.groupby(
    ['DESYNPUF_ID', 'DRUG_MID_3'])['refill_gap_days'].shift(1)

# Flag first fills BEFORE filling NaNs
df_labelled['is_first_fill'] = df_labelled['prev_gap_days'].isna().astype(int)
df_labelled['prev_gap_days'] = df_labelled['prev_gap_days'].fillna(0)

# Define features and target
FEATURES = ['DAYS_SUPLY_NUM', 'QTY_DSPNSD_NUM', 'PTNT_PAY_AMT',
            'TOT_RX_CST_AMT', 'fill_count', 'prev_gap_days', 'is_first_fill']
TARGET = 'is_late'

# Verify
print(df_labelled[FEATURES].isna().sum())
print(f"Rows available: {len(df_labelled):,}")
print(f"First fills: {df_labelled['is_first_fill'].sum():,}")

df_labelled[FEATURES + [TARGET]].head(50)

DAYS_SUPLY_NUM    0
QTY_DSPNSD_NUM    0
PTNT_PAY_AMT      0
TOT_RX_CST_AMT    0
fill_count        0
prev_gap_days     0
is_first_fill     0
dtype: int64
Rows available: 1,142,880
First fills: 826,615


,DAYS_SUPLY_NUM,QTY_DSPNSD_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT,fill_count,prev_gap_days,is_first_fill,is_late
0,30,360.0,0.0,10.0,1,0.0,1,1
1,30,10.0,0.0,160.0,2,255.0,0,1
2,10,30.0,0.0,10.0,1,0.0,1,1
3,30,90.0,0.0,10.0,1,0.0,1,0
4,30,30.0,20.0,280.0,2,-5.0,0,1
5,30,180.0,0.0,10.0,3,578.0,0,0
6,30,90.0,0.0,10.0,4,-27.0,0,0
7,30,30.0,50.0,150.0,5,-29.0,0,0
8,30,30.0,70.0,70.0,1,0.0,1,1
9,30,30.0,70.0,60.0,2,229.0,0,1


In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, classification_report

# Time-based split: train on pre-2010, test on 2010
train = df_labelled[df_labelled['SRVC_DT'] < '2010-01-01']
test  = df_labelled[df_labelled['SRVC_DT'] >= '2010-01-01']

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train size: {len(train):,} | Test size: {len(test):,}")
print(f"Train late %: {y_train.mean():.1%} | Test late %: {y_test.mean():.1%}")

Train size: 1,045,738 | Test size: 97,142
Train late %: 89.8% | Test late %: 71.0%


In [64]:
print(df_labelled['refill_gap_days'].describe())
print(df_labelled['is_late'].value_counts(normalize=True))

count    1.142880e+06
mean     2.301369e+02
std      2.116925e+02
min     -9.000000e+01
25%      5.900000e+01
50%      1.790000e+02
75%      3.590000e+02
max      1.078000e+03
Name: refill_gap_days, dtype: float64
is_late
1    0.882354
0    0.117646
Name: proportion, dtype: float64


In [67]:
# Scale features — important for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"PR-AUC: {pr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, model.predict(X_test_scaled)))

PR-AUC: 0.7914

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.00      0.01     28184
           1       0.71      1.00      0.83     68958

    accuracy                           0.71     97142
   macro avg       0.75      0.50      0.42     97142
weighted avg       0.74      0.71      0.59     97142



In [66]:
# Scale features — important for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"PR-AUC: {pr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, model.predict(X_test_scaled)))

PR-AUC: 0.7919

Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.62      0.45     28184
           1       0.78      0.55      0.64     68958

    accuracy                           0.57     97142
   macro avg       0.57      0.58      0.55     97142
weighted avg       0.66      0.57      0.59     97142



## Experiment: DRUG_MID_4 vs DRUG_MID_3
Testing whether grouping by 4 middle digits improves performance.
Result: DRUG_MID_3 preferred — 5x more data, more robust evaluation.

In [ ]:
#Extracting the middle 4 digits & build df_labelled

# Extract middle 4 digits of drug code
df['DRUG_MID_4'] = df['PROD_SRVC_ID'].astype(str).str[5:9]

# Sort for correct ordering
df_labelled_4 = df.sort_values(
    ['DESYNPUF_ID', 'DRUG_MID_4', 'SRVC_DT']).reset_index(drop=True)

# Calculate expected next fill date
df_labelled_4['EXPECTED_NEXT'] = (
    df_labelled_4['SRVC_DT'] + 
    pd.to_timedelta(df_labelled_4['DAYS_SUPLY_NUM'], unit='D'))

# Calculate actual refill gap
df_labelled_4['next_fill_date'] = df_labelled_4.groupby(
    ['DESYNPUF_ID', 'DRUG_MID_4'])['SRVC_DT'].shift(-1)

df_labelled_4['refill_gap_days'] = (
    df_labelled_4['next_fill_date'] - df_labelled_4['EXPECTED_NEXT']
).dt.days

# Create is_late label
df_labelled_4['is_late'] = (df_labelled_4['refill_gap_days'] > 0).astype(int)

# Drop rows where next fill doesn't exist (last fills)
df_labelled_4 = df_labelled_4.dropna(subset=['refill_gap_days'])

print(f"Rows: {len(df_labelled_4):,}")
print(f"Unique DRUG_MID_4 groups: {df_labelled_4['DRUG_MID_4'].nunique():,}")
print(f"Late %: {df_labelled_4['is_late'].mean():.1%}")

Rows: 185,723
Unique DRUG_MID_4 groups: 4,116
Late %: 92.1%


In [72]:
# Feature engineering + clean NaNs

# Feature 1: prev_gap_days
df_labelled_4['prev_gap_days'] = df_labelled_4.groupby(
    ['DESYNPUF_ID', 'DRUG_MID_4'])['refill_gap_days'].shift(1)

# Feature 2: fill_count
df_labelled_4['fill_count'] = df_labelled_4.groupby(
    ['DESYNPUF_ID', 'DRUG_MID_4']).cumcount() + 1

# Flag first fills before filling NaNs
df_labelled_4['is_first_fill'] = df_labelled_4['prev_gap_days'].isna().astype(int)
df_labelled_4['prev_gap_days'] = df_labelled_4['prev_gap_days'].fillna(0)

FEATURES_4 = ['DAYS_SUPLY_NUM', 'QTY_DSPNSD_NUM', 'PTNT_PAY_AMT',
              'TOT_RX_CST_AMT', 'fill_count', 'prev_gap_days', 'is_first_fill']
TARGET = 'is_late'

print(df_labelled_4[FEATURES_4].isna().sum())
print(f"First fills: {df_labelled_4['is_first_fill'].sum():,}")
print(f"Rows available: {len(df_labelled_4):,}")

DAYS_SUPLY_NUM    0
QTY_DSPNSD_NUM    0
PTNT_PAY_AMT      0
TOT_RX_CST_AMT    0
fill_count        0
prev_gap_days     0
is_first_fill     0
dtype: int64
First fills: 173,022
Rows available: 185,723


In [74]:
# Training & evaluation of the Logistic Regression on DRUG_MID_4 groups

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, classification_report

# Time-based split
train4 = df_labelled_4[df_labelled_4['SRVC_DT'] < '2010-01-01']
test4  = df_labelled_4[df_labelled_4['SRVC_DT'] >= '2010-01-01']

X_train4, y_train4 = train4[FEATURES_4], train4[TARGET]
X_test4,  y_test4  = test4[FEATURES_4],  test4[TARGET]

print(f"Train: {len(train4):,} | Test: {len(test4):,}")
print(f"Train late %: {y_train4.mean():.1%} | Test late %: {y_test4.mean():.1%}")

# Scale + train
scaler4 = StandardScaler()
X_train4_scaled = scaler4.fit_transform(X_train4)
X_test4_scaled  = scaler4.transform(X_test4)

model4 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model4.fit(X_train4_scaled, y_train4)

# Evaluate
y_pred_proba4 = model4.predict_proba(X_test4_scaled)[:, 1]
pr_auc4 = average_precision_score(y_test4, y_pred_proba4)

print(f"\nPR-AUC (DRUG_MID_4): {pr_auc4:.4f}")
print(f"PR-AUC (DRUG_MID_3): 0.7919  ← previous baseline")
print("\nClassification Report:")
print(classification_report(y_test4, model4.predict(X_test4_scaled)))

Train: 172,345 | Test: 13,378
Train late %: 93.3% | Test late %: 76.9%

PR-AUC (DRUG_MID_4): 0.8532
PR-AUC (DRUG_MID_3): 0.7919  ← previous baseline

Classification Report:
              precision    recall  f1-score   support

           0       0.39      0.41      0.40      3093
           1       0.82      0.81      0.81     10285

    accuracy                           0.72     13378
   macro avg       0.60      0.61      0.61     13378
weighted avg       0.72      0.72      0.72     13378



In [75]:
print(f"X_train rows: {len(X_train):,}")   # should be 1,045,738
print(f"X_test rows:  {len(X_test):,}")    # should be 97,142

X_train rows: 1,045,738
X_test rows:  97,142


In [77]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.3/101.7 MB 8.6 MB/s eta 0:00:12
   - -------------------------------------- 3.7/101.7 MB 10.6 MB/s eta 0:00:10
   -- ------------------------------------- 5.5/101.7 MB 9.9 MB/s eta 0:00:10
   -- ------------------------------------- 7.6/101.7 MB 9.7 MB/s eta 0:00:10
   --- ------------------------------------ 9.4/101.7 MB 9.9 MB/s eta 0:00:10
   ---- ----------------------------------- 11.8/101.7 MB 9.9 MB/s eta 0:00:10
   ----- ---------------------------------- 13.9/101.7 MB 10.1 MB/s eta 0:00:09
   ------ --------------------------------- 16.5/101.7 MB 10.4 MB/s eta 0:00:09
   ------- -------------------------------- 18.4/101.7 MB 10.3 MB/s eta 0:00:09
   -------- ------------------------------- 21.0/101.7 MB 10.5 MB/s eta 0:00:08
   --------- ------------------------------ 23.3/101.7 MB 10.7 MB/s eta 0:00:08
   ---------- ----------------------------- 25.4/101.7 MB 1

In [78]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight to handle imbalance (replaces class_weight='balanced')
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"scale_pos_weight: {scale:.2f}")

# Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='aucpr',
    verbosity=0
)
xgb_model.fit(X_train, y_train)  # No scaling needed for XGBoost

# Evaluate
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
pr_auc_xgb = average_precision_score(y_test, y_pred_proba_xgb)

print(f"\nPR-AUC (XGBoost):          {pr_auc_xgb:.4f}")
print(f"PR-AUC (Logistic Reg):     0.7919  ← baseline")
print("\nClassification Report:")
print(classification_report(y_test, xgb_model.predict(X_test)))

scale_pos_weight: 0.11

PR-AUC (XGBoost):          0.8003
PR-AUC (Logistic Reg):     0.7919  ← baseline

Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.61      0.46     28184
           1       0.78      0.56      0.65     68958

    accuracy                           0.58     97142
   macro avg       0.57      0.59      0.56     97142
weighted avg       0.66      0.58      0.60     97142

